this notebook acts the backend for the streamlit based frontend to run. it can run queries directly too.

In [ ]:
#install dependencies
#!pip install langchain_text_splitters langchain_huggingface langchain_chroma google-genai sentence-transformers torch pypdf pdfplumber streamlit python-dotenv

In [ ]:
#import libraries
import os
import pypdf
import pdfplumber
import pandas as pd

### the entire lotr content is huge, even the trilogy can take up to 30mins to load without a gpu.  
i have loaded only a few pages for demonstration

In [ ]:
#load the pdf
lotr_raw = "LOTR.pdf"
with open(lotr_raw, "rb") as f:
    reader = pypdf.PdfReader(f)
    pages = reader.pages[13:62]   #select the pages you want
    texts = [page.extract_text() for page in pages]
    for i, page in enumerate(texts[:3]):
        print(page[:100])

In [ ]:
with open("lotr.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(texts))

In [ ]:
pdf_path = "lotr.txt"

In [ ]:
#clean up
import re

text = "\n".join(texts)
text = re.sub(r'[\t\r\f\v]+', ' ', text)
text = re.sub(r'\n{2,}', '\n', text)
text = re.sub(r' {2,}', ' ', text)
text = text.strip()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
#chunking
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150,
    chunk_overlap=15
)

chunks = text_splitter.split_text(text)
len(chunks)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

In [ ]:
from langchain_core.documents import Document

In [ ]:
#embedding
from langchain_chroma import Chroma
vector_store = Chroma(
    collection_name="lotr_collection",
    embedding_function=embedding_function,
    persist_directory="./lotr_db",
)

In [ ]:
metadata = [{"source": f"page_{i+1}", "book": "lotr", "attempt": 2} for i in range(len(chunks))]
vector_store.add_documents(
    documents=[Document(page_content=text, metadata=meta) for text, meta in zip(chunks, metadata)])

In [ ]:
#for loading the whole book the upper embedding fails so use this instead
# batch_size = 500
# for i in range(0, len(chunks), batch_size):
#     batch_chunks = chunks[i:i+batch_size]
#     batch_meta = metadata[i:i+batch_size]
#     vector_store.add_documents(
#         documents=[Document(page_content=text, metadata=meta) for text, meta in zip(batch_chunks, batch_meta)]
#     )
#     print(f"Added chunks {i} to {i+len(batch_chunks)}")

In [ ]:
#load the llm
from dotenv import load_dotenv
import os
from google import genai
load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
MODEL_NAME = "gemini-2.5-flash"

In [ ]:
#you can run it here directly
question = "Who is Bilbo?"
results = vector_store.similarity_search(
    question,
    k=5,
    filter={"attempt": 2}
)
context = " ".join([res.page_content for res in results])

In [ ]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents=f"""You are a lore expert for The Lord of the Rings trilogy.
Question:
{question}

Context (passages retrieved from the books):
{context}

Instructions:
- Use ONLY the retrieved passages above to answer.
- If the lore is explicitly stated, quote or paraphrase it directly.
- If the context does not contain enough information to answer, say: "The trilogy does not provide enough information to verify this."
- Do NOT use outside knowledge or invent lore — accuracy is critical for a fact-checker.
- Keep the answer concise and cite which part of the context supports it where possible.
- Answer in a warm, knowledgeable tone — like a fellow Tolkien reader explaining lore to a friend, not a machine listing bullet points.
- Avoid bullet points and lists. Answer in flowing prose.
"""
)

In [ ]:
print(response.text)